# RRSIG for ECH active Domains for one Test

In [ ]:
test_date = "2024-11-07"

In [ ]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, test_date, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=(test_date,))
        print(df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}_{test_date}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT COUNT(*) AS signed_ech_config_count
FROM public."HTTPSRecords" hr
WHERE hr.test_date = %s
  AND hr.ech_config IS NOT NULL
  AND hr.ech_config != ''
  AND hr.signed = TRUE;
    """
    filename_suffix = "ech_rrsig_signed_single"
    fetch_and_save_data(query, filename_suffix, test_date)

# RRSIG for ECH active Domains over ECH Domains for all Tests

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-26",
    "2024-08-29",
    "2024-09-01",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-17",
    "2024-10-20",
    "2024-10-27",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
]


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        # Create an empty list to store the dataframes
        df_list = []

        with engine.connect() as conn:
            # Iterate over the testDates list
            for date in testDates:
                # Execute the query for each date
                params = (date,)
                df = pd.read_sql_query(query, conn, params=params)
                df_list.append(df)

        # Concatenate all the dataframes into a single dataframe
        final_df = pd.concat(df_list, ignore_index=True)

        print(final_df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(final_df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    hr.test_date,  -- Use test_date from HTTPSRecords
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.signed = TRUE THEN 1 END) AS signed_ech_config_count
FROM public."HTTPSRecords" hr
WHERE hr.test_date = %s
GROUP BY
    hr.test_date  -- Group by test_date
ORDER BY
    hr.test_date;  -- Order by test_date
    """
    filename_suffix = "ech_rrsig_signed_all_test_dates"
    fetch_and_save_data(query, filename_suffix)

# RRSIG signed and non-signed and total

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=params)
        print(df)

        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    hr.test_date,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' THEN 1 END) AS total_ech_config_count,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.signed = TRUE THEN 1 END) AS signed_ech_config_count,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.signed = FALSE THEN 1 END) AS not_signed_ech_config_count
FROM
    public."HTTPSRecords" hr
GROUP BY
    hr.test_date
ORDER BY
    hr.test_date;
    """
    filename_suffix = "ech_rrsig_signed_all"
    fetch_and_save_data(query, filename_suffix)

# RRSIG Verified for all Tests and Domains

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=params)

        print(df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    hr.test_date,  -- Use test_date from HTTPSRecords
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.verified = TRUE THEN 1 END) AS verified_ech_config_count
FROM public."HTTPSRecords" hr
GROUP BY
    hr.test_date  -- Group by test_date
ORDER BY
    hr.test_date;  -- Order by test_date
    """
    filename_suffix = "ech_rrsig_verified_all"
    fetch_and_save_data(query, filename_suffix)

# RRSIG Errors for all Tests and Domains

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=params)
        print(df)

        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT ar.asn_org, COUNT(DISTINCT dr.domain) AS domain_count
FROM public."DNSResults" dr
JOIN public."DNSResultsARecords" dra ON dr.id = dra.dns_result_id
JOIN public."ARecords" ar ON dra.a_record_id = ar.id
WHERE dr.test_code = '2024-10-13_02-50-30_8.8.8.8'
  AND dr.https_ech_key IS NOT NULL
  AND dr.https_ech_key != ''
GROUP BY ar.asn_org
ORDER BY domain_count DESC;
    """
    filename_suffix = "ech_rrsig_verified_all"
    fetch_and_save_data(query, filename_suffix)

In [ ]:
from multiprocessing import Pool

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    # "2024-08-26",
    "2024-08-29",
    # "2024-09-01",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    # "2024-10-17",
    "2024-10-20",
    # "2024-10-27",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
    "2024-11-15",
    "2024-11-17",
    "2024-11-18",
    "2024-11-29",
]

query = """
SELECT
    hr.test_date,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' THEN 1 END) AS total_ech_config_count,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.signed = TRUE THEN 1 END) AS signed_ech_config_count,
    COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' AND hr.signed = FALSE THEN 1 END) AS not_signed_ech_config_count
FROM
    public."HTTPSRecords" hr
WHERE
    hr.test_date = %s  -- Parameterized test_date
GROUP BY
    hr.test_date
ORDER BY
    hr.test_date;
"""

# Create engine at the module level
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    connect_args={"connect_timeout": 3600},
    poolclass=QueuePool,
    pool_size=5,
    max_overflow=10,
)


def fetch_data(query, date):
    """Fetches data for the given query and date."""
    print(f"Starting query for date: {date}")
    with engine.connect() as conn:
        params = (date,)
        df = pd.read_sql_query(query, conn, params=params)
    print(f"Finished query for date: {date}")
    return date, df


def fetch_and_save_data(query, filename_suffix, testDates, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling and multiprocessing,
    saves each test date's data to separate pickle files, prints all data as a single dataframe,
    and outputs all filenames for easy copying.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        testDates (list): A list of dates to execute the query for.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        # Create a list to store the tasks for the pool
        tasks = [(query, date) for date in testDates]

        # Use a multiprocessing pool to execute the tasks in parallel
        with Pool() as pool:
            results = pool.starmap(fetch_data, tasks)

        all_dfs = []  # List to store all dataframes
        filenames = []  # List to store all filenames

        # Save each date's data to a separate pickle file
        for date, df in results:
            all_dfs.append(df)  # Add the dataframe to the list

            os.makedirs("./pickle", exist_ok=True)
            filename = f"./pickle/{datetime.now().strftime('%Y-%m-%d_%HH-%MM-%SS')}_{filename_suffix}_{date}.pickle"
            filenames.append(filename)  # Add the filename to the list

            with open(filename, "wb") as f:
                pickle.dump(df, f)

        # Concatenate all dataframes and print
        all_data_df = pd.concat(all_dfs, ignore_index=True)
        print("\nAll data:\n", all_data_df)

        # Print all filenames for easy copying
        print("\nFilenames:\n", "\n".join(filenames))

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    filename_suffix = "ech_rrsig_occurrence_count_by_test_date_parallelized"
    fetch_and_save_data(query, filename_suffix, testDates)